# Cocogoat 2026 - Task 5: latent world-GAN - images and videos at 1024x1024 (~10 h)

Inter4K (1,000 real 4K clips, 15 fps) downloaded on Colab. Per-frame beta-VAE at 256 px (1.5 h), then a latent image generator + an adversarial world model trained jointly (5.5 h) with three critics (latent frame, decoded frame, temporal window), then a x4 SR GAN on native 1024 crops (2 h). One system, two modes: world model OFF -> 1024 px images, ON -> 1024 px videos (mp4 + GIF). Text prompts work through CLIP. Outputs: `MyDrive/Cocogoat/runs/task5/`.

**How to run:** `Runtime -> Change runtime type -> G4 (RTX PRO 6000)`, set the budgets in the settings cell (5),
then `Runtime -> Run all`.
Keep this tab open (Colab Pro stops idle/closed sessions). Everything is written to
`MyDrive/Cocogoat/runs/...`; if Colab disconnects, just *Run all* again - training resumes from the last checkpoint.
You do not need to inspect results: metrics, sample grids and logs are saved to Drive for analysis.

In [ ]:
# 1) Mount Google Drive (outputs + datasets live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Get / update the code (public repo, branch claude/cocogoat-revival-glfnmz)
import os, subprocess
REPO, BRANCH = "https://github.com/Martyn0324/Cocogoat.git", "claude/cocogoat-revival-glfnmz"
if not os.path.exists('/content/Cocogoat/.git'):
    subprocess.run(['git', 'clone', '-b', BRANCH, '--depth', '1', REPO, '/content/Cocogoat'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Cocogoat', 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Cocogoat', 'reset', '--hard', 'FETCH_HEAD'], check=True)
print(subprocess.run(['git', '-C', '/content/Cocogoat', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
os.chdir('/content/Cocogoat')

In [ ]:
# 3) Extra packages (torch/torchvision/transformers come with Colab)
!pip -q install -r /content/Cocogoat/requirements-colab.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Colab Secrets are only readable from the notebook kernel: export HF_TOKEN for the training subprocesses
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN exported')
except Exception as exc:
    print('No HF_TOKEN secret (only needed for gated Hugging Face datasets):', type(exc).__name__)
os.environ['PYTHONUNBUFFERED'] = '1'

In [ ]:
# 4) GPU smoke test (tiny models on synthetic clips, always from scratch)
!python -m cocogoat.tasks.task5_world_gan --smoke --run_dir /content/drive/MyDrive/Cocogoat/runs/task5_smoke --force

In [ ]:
# 5) Settings - edit freely. Training time budgets in hours (data/evaluation time not included).
#    MAX_CLIPS: Inter4K clips to use; 0 = all 1,000 (~15-25 min to download and decode).
#    Quick test: e.g. MAX_CLIPS = 100 with small budgets, in a separate folder (e.g. '.../runs/task5_test'):
#    the finished VAE stage is not extended later. (Raising GAN_HOURS / SR_HOURS on the same RUN_DIR continues them.)
MAX_CLIPS, VAE_HOURS, GAN_HOURS, SR_HOURS = 0, 1.5, 5.5, 2.0
RUN_DIR = '/content/drive/MyDrive/Cocogoat/runs/task5'

In [ ]:
# 6) Data, VAE, world-GAN, SR, evaluation + videos (resumes automatically after a disconnect)
!python -m cocogoat.tasks.task5_world_gan --max_clips {MAX_CLIPS} --vae_hours {VAE_HOURS} --gan_hours {GAN_HOURS} --sr_hours {SR_HOURS} --run_dir {RUN_DIR}

In [ ]:
# 7) Results
from IPython.display import Image, Markdown, Video, display
run = RUN_DIR
display(Markdown(open(f'{run}/eval/report.md').read()))
display(Image(f'{run}/eval/samples/prompts_images.png'))
display(Image(f'{run}/eval/videos/prompt0.gif'))